# NumPy
**Topic:** Scientific Python

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


---
## What you'll explore

By the end of this demo you will be able to:

- **Create** NumPy arrays and explain why they are faster than Python lists for numerical operations
- **Apply** broadcasting to perform element-wise operations between arrays of different shapes
- **Interpret** the role of NumPy as the numerical foundation that pandas and scikit-learn are built on

---
## How we got here

In *04: Data Structures* we learned that Python lists can hold anything but are slow for numerical math. In *12: Virtual Environments & Packages* we set up the scientific Python stack. NumPy is the first library in that stack: a typed, fixed-shape array with operations implemented in C that runs 50-100x faster than equivalent Python loops. Every pandas column and every scikit-learn feature matrix is a NumPy array under the hood.

---
## Why this matters for data science

When scikit-learn's `LinearRegression.predict()` computes `X @ self.coef_ + self.intercept_`, it is performing a NumPy matrix multiplication. When pandas computes `.mean()` or `.std()`, it delegates to NumPy. The `@` operator, `np.dot()`, `np.linalg.inv()`, and `np.random` are the mathematical vocabulary of machine learning implementations. You do not need to implement models from scratch, but reading and debugging ML code requires recognizing these operations.

---
## Try it yourself

In [ ]:
# ▶ Run this cell and observe the output.
# Then try changing the values and running again.

import numpy as np

arr = np.array([52000, 75000, 88000, 61000, 94000, 45000])

print('Array:  ', arr)
print('Mean:   ', arr.mean())
print('Std:    ', arr.std().round(2))
print('Max:    ', arr.max())
print('Above 70k:', arr[arr > 70000])

In [ ]:
# ✏️ Your turn — modify this code:
# 1. Change the shape from (4, 3) to (3, 4) — what changes in the output?
# 2. Replace np.zeros with np.ones, then np.arange(12).reshape(4, 3)
# 3. What does matrix.T do? Try printing it.

import numpy as np
np.random.seed(42)

matrix = np.zeros((4, 3))
matrix[:, 0] = [10, 20, 30, 40]   # first column
matrix[:, 1] = [1.1, 2.2, 3.3, 4.4]

print(matrix)
print('Shape:', matrix.shape)
print('Row means:', matrix.mean(axis=1))

In [ ]:
# 🎯 Challenge:
# Create a 5x5 matrix of random integers between 1 and 100, then:
#   1. Find the index of the row with the highest sum
#   2. Normalize the matrix so all values are between 0 and 1
#   3. Replace all normalized values below 0.5 with 0
# Hint: use np.random.randint(), .sum(axis=1), .argmax(), and boolean indexing

import numpy as np
np.random.seed(99)

# Your code here:

---
## What's happening?

A NumPy array is a contiguous block of memory holding elements of a single fixed dtype. This uniformity is what makes element-wise operations fast: no type checking, no pointer chasing, just arithmetic on a flat buffer.

| Concept | Example | What it means |
|---------|---------|--------------|
| Shape | `arr.shape` gives `(3, 4)` | 3 rows, 4 columns |
| Dtype | `arr.dtype` gives `float64` | All elements are 64-bit floats |
| Indexing | `arr[0, 2]` | Row 0, column 2 |
| Slicing | `arr[:, 1]` | All rows, column 1 (returns a view) |
| Broadcasting | `arr + 5` | Adds 5 to every element without a loop |
| Matrix multiply | `A @ B` | Dot product (rows of A times columns of B) |

```python
# The feature-matrix dot-product at the heart of linear models
X      = np.random.randn(100, 5)   # 100 samples, 5 features
theta  = np.array([0.5, -1.2, 0.3, 0.8, -0.4])
y_hat  = X @ theta                  # shape: (100,), one prediction per sample
```

### Broadcasting: NumPy's shorthand for "apply to every element"

When two arrays have compatible shapes, NumPy automatically expands the smaller one to match the larger. A `(100, 5)` feature matrix minus a `(5,)` mean vector subtracts each feature's mean across all 100 rows, zero copy, zero loop.

Return to the widget and step through the broadcasting example: verify the output shape before looking at the result.

---
## A direct example: normalizing an array with broadcasting

Eight raw values, one formula, no loop. NumPy broadcasts `(raw - raw.min()) / (raw.max() - raw.min())` across every element at once, scaling all values into the range [0, 1].

- **Notice:** `raw.min()` and `raw.max()` return scalars; NumPy automatically expands them to match the shape of `raw` — no loop needed
- **Notice:** The same two lines that work on 8 values work identically on 8 million values
- **Notice:** This is min-max normalization — the simpler sibling of the z-score standardization shown in the real-world example below

In [ ]:
labels = ["S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8"]
raw    = np.array([23, 85, 41, 67, 12, 95, 58, 34])

normalized = (raw - raw.min()) / (raw.max() - raw.min())

print("Raw:       ", raw)
print("Normalized:", normalized.round(2))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(labels, raw, color="#E45756")
axes[0].set_title("Raw values", fontsize=13)
axes[0].set_ylabel("Value")
axes[0].spines[["top", "right"]].set_visible(False)

axes[1].bar(labels, normalized, color="#4C72B0")
axes[1].set_title("After (raw - raw.min()) / (raw.max() - raw.min())", fontsize=12)
axes[1].set_ylabel("Normalized value [0, 1]")
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

---
## Real-world example: Standardizing a feature matrix for machine learning

Before training distance-based models (k-NN, SVM, k-means), features must be standardized to zero mean and unit variance. This is exactly what `StandardScaler` does, and it is three NumPy operations.

The heatmap below shows a 5-feature matrix before and after standardization. Notice:

- **Notice:** The raw matrix has columns on very different scales (age in tens, salary in thousands); the standardized matrix has every column in the same range, which is required for distance-based models
- **Notice:** Standardization is two broadcasts: subtract the column mean (shape `(5,)` broadcast over `(200, 5)`) then divide by the column standard deviation (same broadcast)
- **Notice:** The standardized values have mean ≈ 0 and std ≈ 1 per column, which you can verify by calling `np.mean(X_std, axis=0)` and `np.std(X_std, axis=0)`

> **Discussion question:** If you accidentally compute the mean and std from the test set (rather than the training set) before standardizing, what goes wrong statistically? Why does sklearn's `StandardScaler` require you to call `fit()` on training data only?

In [ ]:
np.random.seed(42)

# ── Feature matrix standardization using pure NumPy ───────────────────────────
n, p      = 200, 5
col_names = ["age", "income", "score", "tenure", "spend"]

X_raw = np.column_stack([
    np.random.randint(22, 65, n).astype(float),
    np.random.lognormal(10.8, 0.4, n),
    np.random.uniform(300, 850, n),
    np.random.exponential(4, n),
    np.random.lognormal(6.5, 0.8, n),
])

mu    = X_raw.mean(axis=0)
sigma = X_raw.std(axis=0)
X_std = (X_raw - mu) / sigma

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

im0 = axes[0].imshow(X_raw[:20], aspect="auto", cmap="RdBu_r")
axes[0].set_xticks(range(p))
axes[0].set_xticklabels(col_names)
axes[0].set_title("Raw Feature Matrix (first 20 rows)")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(X_std[:20], aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3)
axes[1].set_xticks(range(p))
axes[1].set_xticklabels(col_names)
axes[1].set_title("Standardized Feature Matrix (first 20 rows)")
plt.colorbar(im1, ax=axes[1])

fig.suptitle("NumPy Standardization: (X - mean) / std — Broadcasting in Action",
             fontsize=13)
plt.tight_layout()
plt.show()


### NumPy operations every data scientist uses

| Operation | Code | What it does |
|-----------|------|-------------|
| Create array | `np.array([1, 2, 3])` | Convert a list to a typed array |
| Random data | `np.random.randn(n, p)` | n×p matrix of standard normals |
| Element-wise math | `X ** 2`, `np.log(X)`, `np.sqrt(X)` | Apply operation to every element |
| Aggregation | `X.mean(axis=0)`, `X.sum(axis=1)` | Collapse along a dimension |
| Matrix multiply | `A @ B`, `np.dot(A, B)` | Row × column dot product |
| Concatenate | `np.hstack([A, B])`, `np.vstack([A, B])` | Join arrays along an axis |
| Boolean indexing | `X[X > 0]` | Select elements where condition is true |

---
## Key takeaway

> **NumPy arrays store a single dtype in contiguous memory, which makes arithmetic 50-100x faster than Python lists; every pandas column and scikit-learn feature matrix is a NumPy array, so understanding arrays means understanding the entire scientific Python stack.**

---
*Next up: Pandas — the DataFrame library that wraps NumPy arrays into labeled, column-oriented tables and gives you the data manipulation tools you will use every single day*